# Lab 5 · Measure first: an evaluation set and the hallucination baseline

**Time:** 25 min · **You will:** build the yardstick for the rest of the course — a held-out test set, an automatic scorer, and a baseline score — and see how badly a plain SLM hallucinates on company-specific questions.

> **Rule of thumb:** never change a model, prompt or pipeline without a number that tells you whether it got better.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**, read each result, then change the `🔧` values and run those cells again (Shift+Enter).*

In [ ]:
import sys, os
# find the course folder (the one holding slm_lab.py), whichever folder this notebook was opened from
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import *
import pandas as pd

items = eval_items()
df = pd.DataFrame(items)
print(len(items), "test questions")
df.groupby("kind").size()

## 5.1 The test set
`data/hr_facts.json` has **30 topics**: 22 facts from the HR policy and 8 questions the policy does **not** answer (gym, bonus, stock options…). Each topic has *training* questions (used in Extra Lab B–7) and **held-out evaluation** questions, worded differently, that are never trained on.

The right answer to an unknown topic is to say so and point to the HR helpdesk — not to invent a policy.

In [ ]:
df.sample(6, random_state=0)[["id", "kind", "question", "expect"]]

## 5.2 The scorer
`is_correct(answer, expect)` passes if the answer contains **every** expected fact; `|` separates acceptable alternatives. It ignores case and thousands separators.

### 🔧 Try it 5.1 — Break the scorer
Automatic scorers are never perfect. For the paternity-leave question (`expect=["20"]`), the cell scores two answers:
- one that is **wrong but passes** (it mentions "20" for another reason),
- one that is **right but fails** (it spells the number out).

🔧 **Try:** write your own answers in the quotes and see whether the scorer is fooled.

In [ ]:
expect = ["20"]
wrong_but_passes = "Paternity leave is 10 days, as decided in 2020."   # 🔧 a wrong answer that contains "20"
right_but_fails = "You get twenty days of paternity leave."            # 🔧 a right answer without "20"
print("wrong answer scored as correct?", is_correct(wrong_but_passes, expect))
print("right answer scored as correct?", is_correct(right_but_fails, expect))

In [ ]:
check(is_correct(wrong_but_passes, expect) and not is_correct(right_but_fails, expect),
      "Exactly. Keyword scorers give false passes (the number appears for another reason) and false fails "
      "(the same fact phrased differently). Always read a sample of results by hand, not just the score.",
      "the first answer should contain '20' somewhere; the second should say the number in words.")

## 5.3 The baseline: a plain SLM as the HR assistant

In [ ]:
tok, model = load()
SYSTEM = HR_SYSTEM
print(SYSTEM)

### 👀 See it 5.2 — Score the plain model
`ask_plain(questions)` answers a list of questions with the plain model (greedy, up to 80 tokens), and `run_eval` scores every answer on the test set.

In [ ]:
def ask_plain(questions):
    return chat(model, tok, questions, system=SYSTEM, max_new_tokens=80)

acc_plain, res_plain = run_eval(ask_plain)
print(f"Plain SLM accuracy: {acc_plain:.0%}")
res_plain.groupby("kind")["correct"].mean()

In [ ]:
check(len(res_plain) == len(items), f"Baseline recorded: {acc_plain:.0%}.",
      "not every test question was answered - run the cell above again.")
save_report("plain", {"accuracy": round(float(acc_plain), 3),
                      "fact_accuracy": round(float(res_plain[res_plain.kind == "fact"].correct.mean()), 3),
                      "unknown_accuracy": round(float(res_plain[res_plain.kind == "unknown"].correct.mean()), 3)})

## 5.4 Read the failures
A score tells you *how much* is wrong; reading tells you *what kind* of wrong.

In [ ]:
pd.set_option("display.max_colwidth", 140)
res_plain[~res_plain.correct][["question", "answer"]].head(12)

### 👀 See it 5.3 — Classify the errors
The cell sorts the first 12 wrong answers with a simple rule:
- **hallucinated** — contains a number (a specific figure stated as fact, with no basis),
- **evasive** — refuses or says "it depends / check with HR" without the fact,
- **other** — off-topic, wrong format, scorer error.

Read the table above: do you agree with the rule? Which answers would you classify differently?

In [ ]:
import re
wrong = res_plain[~res_plain.correct].head(12)

def classify(answer):
    if re.search(r"\d", answer):
        return "hallucinated"
    if re.search(r"depend|contact|check with|consult|refer to|not (able|sure)|cannot", answer, re.I):
        return "evasive"
    return "other"

error_types = {k: int(v) for k, v in wrong["answer"].map(classify).value_counts().items()}
error_types

In [ ]:
check(sum(error_types.values()) > 0,
      "Most failures on fact questions are confident made-up numbers, and on unknown topics the model invents a "
      "policy instead of saying it doesn't know. That's what Labs 5–7 fix.",
      "there were no wrong answers to classify.")

## 🧠 Check your understanding
1. Why are the evaluation questions worded differently from the training questions?
2. The plain model scores low here but is a perfectly good model. What is it missing?
3. Why is the "unknown" category in the test set important for an HR assistant?

*✍️ Your answer:* 

**Next:** Lab 6 — RAG: give the model the policy at question time.